In [ ]:
categories = {
    "Returns & Refunds" : ["Return Request", "Refund Pending"],
    "Shipping & Delivery" : ["Delivery Delayed", "Tracking Issue" , "Package Missing"],
    "Product Issue" : ["Wrong Item", "Wrong Size", "Wrong Color", "Damaged Product", "Defective Product"],
    "Payment & Billing" : ["Duplicate Charge", "Payment Failed", "Unexpected Charge"],
    "Account & Login" : ["Login Problem", "Account Access"],
    "Order Changes" : ["Cancel Order", "Change Order"],
    "Other" : ["Good Review"]
    }


In [ ]:
json_schema = {
    "type": "object",
    "properties": {
        "examples": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string"
                    },
                    "ticket": {
                        "type": "object",
                        "properties": {
                            "category": {
                                "type": "string"
                            },
                            "subcategory": {
                                "type": "string"
                            },
                            "urgency": {
                                "type": "integer"
                            },
                            "sentiment": {
                                "type": "string",
                                "enum": [
                                    "POSITIVE",
                                    "NEUTRAL",
                                    "FRUSTRATED",
                                    "ANGRY"
                                ]
                            },
                            "customer_satisfaction": {
                                "type": "string",
                                "enum": [
                                    "SATISFIED",
                                    "NEUTRAL",
                                    "DISSATISFIED",
                                    "UNKNOWN"
                                ]
                            },
                            "detected_product": {
                                "type": ["string", "null"]
                            },
                            "issue": {
                                "type": "string"
                            },
                            "requested_action": {
                                "type": "string",
                                "enum": [
                                    "Refund",
                                    "Return",
                                    "Exchange",
                                    "Replacement",
                                    "Cancel Order",
                                    "Order Update",
                                    "Account Assistance",
                                    "Payment Resolution",
                                    "Information",
                                    "No Action",
                                    "Other"
                                ]
                            },
                            "requires_human": {
                                "type": "boolean"
                            }
                        },
                        "required": [
                            "category",
                            "subcategory",
                            "urgency",
                            "sentiment",
                            "customer_satisfaction",
                            "detected_product",
                            "issue",
                            "requested_action",
                            "requires_human"
                        ],
                        "additionalProperties": False
                    }
                },
                "required": [
                    "email",
                    "ticket"
                ],
                "additionalProperties": False
            },
            "minItems": 4,
            "maxItems": 4
        }
    },
    "required": ["examples"],
    "additionalProperties": False
}







def prompt(category, sub_category, n):
    
    prompt = f"""
        Generate customer support emails and their structured tickets using the
        provided Category and Subcategory exactly.

        Category : {category}
        Sub-category = {sub_category}

        Rules:

        1. Category and Subcategory are provided with every request.
        Use them exactly as provided. The email must clearly match them.

        2. detected_product:
        - If a product is explicitly mentioned anywhere in the email, detected_product MUST contain that product.
        - If no product is explicitly mentioned anywhere in the email, detected_product MUST be null.
        - Never set detected_product to null when a product is mentioned.
        - Never invent or infer a product that is not mentioned.

        3. Wrong Item vs Wrong Size vs Wrong Color (Keep in mind when the sub-category mentions on of these):
        - Wrong Item = the customer received a completely different product than the one ordered.
        - Wrong Size = the correct product was received, but the size is incorrect.
        - Wrong Color = the correct product was received, but the color is incorrect.

        4. Damaged Product vs Defective Product(keep in mind when the sub-category mentions one of these):
        - Damaged Product = the product arrived physically damaged, broken, cracked, torn, dented, or otherwise damaged during delivery/handling.
        - Defective Product = the product is physically intact but does not work correctly, malfunctions, or has a manufacturing/performance problem.
        
        5. Information vs Order Update(required_action)
        . "Information" = asking for information, status, explanation, or an update about an order; "Order Update" = explicitly requesting a change to existing order details.
        . Words like "update", "status", or "order" alone must NEVER trigger "Order Update".
        . In short: asking to know something → Information; asking to change something → Order Update.
    
        6. sentiment:
        Use only: POSITIVE, NEUTRAL, FRUSTRATED, ANGRY.
        Determine it from the customer's actual tone.

        7. customer_satisfaction:
        Use only: SATISFIED, NEUTRAL, DISSATISFIED, UNKNOWN.
        Do not assume dissatisfaction simply because the customer has a problem.

        8. issue:
        Give a very brief description of the customer's actual concern.
        Only use information present in the email. Never invent details.

        9. requested_action:
        Use only:
        Refund, Return, Exchange, Replacement, Cancel Order,
        Order Update, Account Assistance, Payment Resolution,
        Information, No Action, Other.

        If the customer asks for information about a return, refund, delivery,
        payment, account, etc., use Information.

        If the customer explicitly requests an action, use that action.
        Do not infer what support should do.

        "Information" = asking for information, status, explanation, or an update about an order;
        "Order Update" = explicitly requesting a change to existing order details. this comes under the category "Order Changes"
         
        In short: asking to know something → Information; asking to change something → Order Update.

        10. urgency:
        Score from 1–10.
        1 = no urgency
        3 = mildly time-sensitive
        5 = moderately urgent
        7 = highly urgent
        10 = critical/immediate attention

        Consider deadlines, financial impact, security risk, missing items,
        delays, and explicit time pressure. Do not base urgency on emotion alone.

        11. requires_human:
        true when investigation, verification, authorization, manual correction,
        or human intervention is required.
        false when the request can reasonably be handled automatically.

        12. The ticket must be completely supported by the email.
        Never invent order numbers, products, actions, events, or other details.

        13. Return ONLY valid JSON containing an "examples" array with exactly {n}
            objects. No explanations or text outside the JSON.
        """

    return prompt
    


In [ ]:
import json

def save(data):
    with open("raw_emails.jsonl", "a", encoding= "utf-8") as f:
        f.write(json.dumps(data) + "\n")

In [ ]:
import os 
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("GROQ_API_KEY")

# cs = [ "NEUTRAL", "DISSATISFIED", "UNKNOWN"]
import requests
import json
import time

for i in range(3):
      print("=" * 70)
      print(f"Iteration no : {i}")
      for category, sub_categories in categories.items():
        print(f"Starting Category : {category}")
        for sub_category in sub_categories:
                print(f"Starting Sub-category : {sub_category}")
            
                headers = {
                "Authorization" : f"Bearer {API_KEY}",
                "Content-Type" : "application/json"
            }

                request_body = {
                "model" : "openai/gpt-oss-20b" ,
                "messages" : [
                    {
                        "role" : "system", 
                        "content" : prompt(category , sub_category, 4 )
                    }],
                "temperature" : 0.7,
                "response_format": {
                                        "type": "json_schema",
                                        "json_schema": {
                                            "name": "Email",
                                            "strict": True,
                                            "schema":  json_schema
                                    }
                                        }
                    
            }

                URL = "https://api.groq.com/openai/v1/chat/completions"

                response = requests.post(
                url= URL,
                headers = headers,
                json= request_body
            )

                if response.status_code == 200:

                    data = response.json()
                    info = data["choices"][0]["message"]["content"]
                    parsed_info = json.loads(info)
                    for example in parsed_info["examples"]:
                        save(example)
                    print(f"{sub_category} Done!")
                    time.sleep(20)

                elif response.status_code == 429:
                    print("RATE LIMITED")
                    print("Retry-After:", response.headers.get("retry-after"))
                    print("Response:", response.text)
                    time_needed = response.headers.get("retry-after")

                    if time_needed:
                        time.sleep(int(time_needed))
                    else:
                        time.sleep(10)

                    continue
                else:
                    print(f"API FAILURE: {response.status_code}")
                    print(response.text)

        print(f"{category} Done!")


In [ ]:
# Viewing The Data Collected

import json

with open("raw_emails.jsonl", "r", encoding= "utf-8") as f:
    for data in f:
        data = json.loads(data)
        print("="*70)
        print(f"Email : {data["email"]} \n")
        for  (k,v) in (data['ticket'].items()):
            print(f"{k} : {v}\n")
        

In [ ]:
# Formatting the dataset 

def format_samples(sample):

    dict = {
        "messages" : [
            {
                "role" : "system", "content" : "You Classify Customer Support Emails into a structured JSON tickets"
            },
            {
                "role" : "user", "content" : json.dumps(sample["email"])
            },
            {
                "role" : "assistant", "content" : json.dumps(sample["ticket"])
            }
        ]
    }

    with open("emails.jsonl", "a", encoding= "utf-8") as f:
        f.write(json.dumps(dict) + "\n")

In [ ]:
with open("raw_emails.jsonl", "r", encoding= "utf-8") as f:
    for sample in f:
        sample = json.loads(sample)
        format_samples(sample)
